# Topic Modeling: French Legislative Election Manifestos

This notebook explores topic modeling on French legislative election manifestos using different methods.
Starting with **LDA (Latent Dirichlet Allocation)** on 1973 data.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Topic modeling
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# Visualization
import pyLDAvis
import pyLDAvis.lda_model

# Set style
plt.style.use('default')
sns.set_palette("husl")

print("All libraries imported successfully!")

In [ ]:
import spacy
! python -m spacy download fr_core_news_sm

## 2. Stopwords and lemmatization

In [ ]:
STOPWORDS = [x.strip() for x in open('data/stop_word_fr.txt').readlines()]
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])


## 3. Load Metadata and Map Text Files to Candidates

In [ ]:
# Load the CSV data
df = pd.read_csv('archelect_search.csv')

# Extract year and filter
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
df = df[df['contexte-election'] == "législatives"]

# Focus on 1973 for initial testing
year_of_interest = 1973
df_1973 = df[df['year'] == year_of_interest].copy()

print(f"Candidates in {year_of_interest}: {len(df_1973)}")
print(f"Columns available: {list(df.columns[:10])}...")
print(f"\nFirst few rows:")
df_1973.head()

## 4. Load Text Files for 1973

In [ ]:
# Collect text files from 1973
text_dir = 'text_files/1973/legislatives'
text_files = []

if os.path.exists(text_dir):
    for file in os.listdir(text_dir):
        if file.endswith('.txt'):
            text_files.append(os.path.join(text_dir, file))

print(f"Found {len(text_files)} text files from {year_of_interest}")

# Load all texts
documents = []
file_names = []

for file_path in text_files:
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            documents.append(content)
            file_names.append(os.path.basename(file_path))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")

print(f"\nLoaded {len(documents)} documents")
print(f"Average document length: {np.mean([len(doc.split()) for doc in documents]):.0f} words")
print(f"Min length: {min([len(doc.split()) for doc in documents])} words")
print(f"Max length: {max([len(doc.split()) for doc in documents])} words")

## 5. French Text Preprocessing

In [ ]:
# French stop words and preprocessing
french_stopwords = set(stopwords.words('french'))

# Add common French political terms that might not be informative
additional_stopwords = {'loi', 'article', 'gouvernement', 'français', 'france', 
                        'monsieur', 'madame', 'nombre', 'partie', 'pays'}
french_stopwords.update(additional_stopwords)

# French stemmer
stemmer = SnowballStemmer('french')

def preprocess_text(text):
    """
    Preprocess French text:
    1. Convert to lowercase
    2. Remove URLs and special characters
    3. Tokenize
    4. Remove stopwords
    5. Stem words
    6. Keep only words with 3+ characters
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-zàâäéèêëïîôöùûüœæ\s]', ' ', text)
    
    # Tokenize
    tokens = text.split()
    
    # Remove stopwords and short words, apply stemming
    tokens = [stemmer.stem(word) for word in tokens 
              if word not in french_stopwords and len(word) >= 3]
    
    return ' '.join(tokens)

# df['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(df['text'])]

print("Processing texts...")
processed_documents = [preprocess_text(doc) for doc in documents]

# Show example of preprocessing
print(f"\n--- PREPROCESSING EXAMPLE ---")
print(f"Original (first 300 chars):\n{documents[0][:300]}")
print(f"\nProcessed (first 300 chars):\n{processed_documents[0][:300]}")

## 6. Create Document-Term Matrix with CountVectorizer

## 7. Train LDA Model

In [ ]:
# Train LDA with different numbers of topics to find optimal
# Starting with 5 topics as a baseline
n_topics = 5
n_top_words = 15

print(f"Training LDA model with {n_topics} topics...")

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online',
    n_jobs=-1,
    verbose=0
)

lda_model.fit(doc_term_matrix)

print(f"Model trained! Perplexity: {lda_model.perplexity(doc_term_matrix):.4f}")

## 8. Display Top Words per Topic

In [ ]:
def display_topics(model, feature_names, n_top_words=15):
    """
    Display the top words for each topic
    """
    print(f"\n{'='*80}")
    print(f"TOP {n_top_words} WORDS PER TOPIC (LDA with {model.n_components} topics)")
    print(f"{'='*80}\n")
    
    for topic_idx, topic in enumerate(model.components_):
        top_words_idx = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_words_idx]
        weights = topic[top_words_idx]
        
        print(f"Topic {topic_idx + 1}:")
        for word, weight in zip(top_words, weights):
            print(f"  {word:20s} {weight:.4f}")
        print()

display_topics(lda_model, vocab, n_top_words)

## 9. Analyze Topic Distribution per Document

## 10. Visualizations

In [ ]:
# Plot 1: Topic Distribution
fig, ax = plt.subplots(figsize=(12, 6))

topic_counts = doc_topic_df['Dominant_Topic'].value_counts().sort_index()
colors = sns.color_palette('husl', n_topics)

ax.bar(range(n_topics), [topic_counts.get(f'Topic_{i+1}', 0) for i in range(n_topics)], color=colors)
ax.set_xlabel('Topic', fontsize=12)
ax.set_ylabel('Number of Documents', fontsize=12)
ax.set_title(f'Document Distribution across Topics (1973) - {n_topics} Topics', fontsize=14, fontweight='bold')
ax.set_xticks(range(n_topics))
ax.set_xticklabels([f'Topic {i+1}' for i in range(n_topics)])
plt.tight_layout()
plt.show()

# Plot 3: Topic composition in sample documents
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx in range(min(6, len(doc_topic_dist))):
    topics = [f'Topic {i+1}' for i in range(n_topics)]
    values = doc_topic_dist[idx]
    colors = sns.color_palette('husl', n_topics)
    
    axes[idx].bar(topics, values, color=colors)
    axes[idx].set_ylabel('Probability', fontsize=10)
    axes[idx].set_title(f'Document {idx+1}: {file_names[idx][:50]}...', fontsize=10, fontweight='bold')
    axes[idx].set_ylim([0, 1])
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 11. Interactive Visualization with pyLDAvis

## 12. Export Results

## 13. Model Evaluation and Comparison (Optional - for testing multiple topic numbers)